# 🛰️ DisasterVision – Building Damage Assessment using Siamese CNN

**Dataset:** xBD / xView2 (pre/post disaster satellite imagery with building polygon annotations)  
**Model:** Siamese ResNet18 (shared backbone) + feature fusion + 4-class damage classifier  
**Classes:** No Damage · Minor Damage · Major Damage · Destroyed  

---
**Execution order:** Run all cells in sequence. Only edit the **Configuration** cell (Section 5).

## Section 1 – Environment Setup

In [ ]:
# Install / upgrade required packages
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q opencv-python-headless scikit-learn matplotlib seaborn tqdm Pillow
print('Packages installed.')

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name}  |  Memory: {gpu.total_memory/1e9:.1f} GB')
    device = torch.device('cuda')
else:
    print('WARNING: No GPU detected – training will be slow on CPU')
    device = torch.device('cpu')
print('Device:', device)

## Section 2 – Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

## Section 3 – Clone Project Repository

Replace `YOUR_USERNAME` with your GitHub username, or upload the project files manually.

In [ ]:
import os, sys

PROJECT_DIR = '/content/DisasterVision'

if not os.path.exists(PROJECT_DIR):
    # Option A: clone from GitHub
    # !git clone https://github.com/YOUR_USERNAME/DisasterVision.git {PROJECT_DIR}
    
    # Option B: copy from Drive (if you uploaded the zip)
    # !unzip /content/drive/MyDrive/DisasterVision.zip -d /content/
    
    # For this demo: create the directory structure inline
    os.makedirs(PROJECT_DIR, exist_ok=True)
    print('Created PROJECT_DIR – paste project files here or uncomment a clone option above.')
else:
    print('Project directory already exists.')

%cd {PROJECT_DIR}
sys.path.insert(0, PROJECT_DIR)
print('Working directory:', os.getcwd())

## Section 4 – Define Project Modules Inline

If the project files are already cloned/uploaded, skip this section and go to Section 5.  
This section defines all required classes inline for a self-contained notebook.

In [ ]:
# ── Annotation Parser (inline) ───────────────────────────────────────────────
import json
import logging
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import numpy as np

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('DisasterVision')

XBD_LABEL_MAP = {
    'no-damage': 0, 'minor-damage': 1, 'major-damage': 2,
    'destroyed': 3, 'un-classified': None,
}
CLASS_NAMES = ['No Damage', 'Minor Damage', 'Major Damage', 'Destroyed']
DAMAGE_HEX  = {0:'#2ecc71', 1:'#f1c40f', 2:'#e67e22', 3:'#e74c3c'}

def polygon_to_bbox(pts):
    return int(pts[:,0].min()), int(pts[:,1].min()), int(pts[:,0].max()), int(pts[:,1].max())

def parse_annotation_file(json_path, img_w=1024, img_h=1024, label_map=None):
    if label_map is None: label_map = XBD_LABEL_MAP
    data = json.loads(Path(json_path).read_text())
    features = data.get('features', {}).get('lng_lat', [])
    buildings = []
    for feat in features:
        props   = feat.get('properties', {})
        geom    = feat.get('geometry', {})
        uid     = props.get('uid', 'unknown')
        subtype = props.get('subtype', 'un-classified').strip().lower()
        coords  = geom.get('coordinates', [])
        if geom.get('type') != 'Polygon' or not coords: continue
        pts = np.array(coords[0], dtype=np.float32)
        pts[:,0] = np.clip(pts[:,0], 0, img_w)
        pts[:,1] = np.clip(pts[:,1], 0, img_h)
        pts = pts.astype(np.int32)
        if len(pts) < 3: continue
        x1,y1,x2,y2 = polygon_to_bbox(pts)
        if (x2-x1)<4 or (y2-y1)<4: continue
        damage_class = label_map.get(subtype, None)
        buildings.append({'uid':uid,'polygon':pts,'bbox':(x1,y1,x2,y2),
                          'damage_label':subtype,'damage_class':damage_class})
    return buildings

def parse_image_pair(pre_json, post_json, img_w=1024, img_h=1024):
    pre_map  = {b['uid']: b for b in parse_annotation_file(pre_json,  img_w, img_h)}
    post_map = {b['uid']: b for b in parse_annotation_file(post_json, img_w, img_h)}
    matched = []
    for uid, post in post_map.items():
        if post['damage_class'] is None: continue
        pre = pre_map.get(uid)
        if pre is None: continue
        matched.append({'uid':uid,'pre_polygon':pre['polygon'],'post_polygon':post['polygon'],
                        'pre_bbox':pre['bbox'],'post_bbox':post['bbox'],
                        'damage_label':post['damage_label'],'damage_class':post['damage_class']})
    return matched

print('Annotation parser defined.')

In [ ]:
# ── Siamese CNN (inline) ─────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

class SharedResNet18(nn.Module):
    FEATURE_DIM = 512
    def __init__(self, pretrained=True):
        super().__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        base    = models.resnet18(weights=weights)
        self.features = nn.Sequential(*list(base.children())[:-1])
    def forward(self, x):
        return self.features(x).flatten(1)

class SiameseDamageClassifier(nn.Module):
    def __init__(self, num_classes=4, dropout=0.4, pretrained=True, freeze_backbone=False):
        super().__init__()
        self.backbone = SharedResNet18(pretrained)
        if freeze_backbone:
            for p in self.backbone.parameters(): p.requires_grad = False
        feat_dim   = SharedResNet18.FEATURE_DIM
        fusion_dim = feat_dim * 3
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512), nn.BatchNorm1d(512),
            nn.ReLU(True), nn.Dropout(dropout),
            nn.Linear(512, 256), nn.BatchNorm1d(256),
            nn.ReLU(True), nn.Dropout(dropout/2),
            nn.Linear(256, num_classes),
        )
        self.num_classes = num_classes

    def forward(self, pre, post):
        f_pre  = self.backbone(pre)
        f_post = self.backbone(post)
        diff   = torch.abs(f_pre - f_post)
        fused  = torch.cat([f_pre, f_post, diff], dim=1)
        return self.classifier(fused)

    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True

    @torch.no_grad()
    def predict(self, pre, post):
        self.eval()
        logits = self(pre, post)
        probs  = F.softmax(logits, dim=1)
        return probs.argmax(1), probs

print('SiameseDamageClassifier defined.')

# Quick architecture test
_model = SiameseDamageClassifier(num_classes=4, pretrained=False)
_pre   = torch.randn(2, 3, 128, 128)
_post  = torch.randn(2, 3, 128, 128)
_out   = _model(_pre, _post)
print(f'Forward pass OK: input {list(_pre.shape)} → logits {list(_out.shape)}')
del _model, _pre, _post, _out

## Section 5 – ⚙️ Configuration

**Edit only this cell.** All other cells read from these variables.

In [ ]:
import os
from pathlib import Path

# ── Dataset path ─────────────────────────────────────────────────────────────
# RECOMMENDED DOWNLOAD: Challenge training set (~7.8 GB)
#   URL: https://xview2.org/dataset  (requires registration)
#   File: train_images_labels_targets.tar.gz
#   SHA1: b37a4ef4ee9c909e2b19d046e49d42ee3965714b
#
# After extracting you get:
#   train/images/<event>_<tile>_pre_disaster.png
#   train/labels/<event>_<tile>_pre_disaster.json
#
# Upload the extracted "train/" folder to Google Drive at:
#   My Drive/xBD/train/
# Then set XBD_ROOT = '/content/drive/MyDrive/xBD/train'
XBD_ROOT = '/content/drive/MyDrive/xBD/train'
os.environ['XBD_ROOT'] = XBD_ROOT

# ── Output directories ───────────────────────────────────────────────────────
OUTPUT_DIR     = Path('/content/outputs');   OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR = Path('/content/checkpoints'); CHECKPOINT_DIR.mkdir(exist_ok=True)
DRIVE_SAVE_DIR = Path('/content/drive/MyDrive/DisasterVision')
try: DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
except: print('Drive not mounted - checkpoints will only be saved locally')

# ── Hyper-parameters ─────────────────────────────────────────────────────────
IMAGE_SIZE           = 128
BATCH_SIZE           = 32
NUM_EPOCHS           = 10
LEARNING_RATE        = 1e-4
WEIGHT_DECAY         = 1e-5
DROPOUT_RATE         = 0.4
EARLY_STOP_PATIENCE  = 4
TRAIN_RATIO          = 0.70
VAL_RATIO            = 0.15
RANDOM_SEED          = 42
NUM_CLASSES          = 4

print(f'XBD_ROOT     : {XBD_ROOT}')
print(f'IMAGE_SIZE   : {IMAGE_SIZE}')
print(f'BATCH_SIZE   : {BATCH_SIZE}')
print(f'NUM_EPOCHS   : {NUM_EPOCHS}')
print(f'DEVICE       : {device}')


## Section 6 – xBD Dataset Inspection

In [ ]:
import os, re, json as _json
from pathlib import Path

xbd_root = Path(XBD_ROOT)
if not xbd_root.exists():
    print(f'ERROR: xBD dataset not found at {XBD_ROOT}')
    print()
    print('To obtain xBD (use Challenge training set, NOT the 51 GB GeoTIFF):')
    print('  1. Register at https://xview2.org/dataset')
    print('  2. Download  train_images_labels_targets.tar.gz  (~7.8 GB)')
    print('     SHA1: b37a4ef4ee9c909e2b19d046e49d42ee3965714b')
    print('  3. Download  test_images_labels_targets.tar.gz   (~2.6 GB)')
    print('     SHA1: 86ed3dba2f8d16ceceb75d451005054fefa9616f')
    print('  4. Extract both archives:')
    print('       tar -xvzf train_images_labels_targets.tar.gz')
    print('       tar -xvzf test_images_labels_targets.tar.gz')
    print('  5. Upload extracted train/ folder to Google Drive at:')
    print(f'     {XBD_ROOT}')
    raise FileNotFoundError(f'xBD not found at {XBD_ROOT}')

# Auto-detect layout
flat_images = xbd_root / 'images'
flat_labels = xbd_root / 'labels'
layout = 'flat' if (flat_images.is_dir() and flat_labels.is_dir()) else 'event_subdirs'
print(f'Detected layout : {layout}')

# Count tiles and annotations
post_jsons = sorted(xbd_root.rglob('*_post_disaster.json'))
pre_imgs   = list(xbd_root.rglob('*_pre_disaster.png')) + list(xbd_root.rglob('*_pre_disaster.tif'))
post_imgs  = list(xbd_root.rglob('*_post_disaster.png')) + list(xbd_root.rglob('*_post_disaster.tif'))

def extract_event(stem):
    m = re.match(r'^(.+?)_(\d{8})$', stem)
    return m.group(1) if m else stem.split('_')[0]

if layout == 'flat':
    events = set(extract_event(p.stem.replace('_post_disaster', '')) for p in post_jsons)
else:
    events = set(p.parts[-3] for p in post_jsons if len(p.parts) >= 3)

print(f'Disaster events      : {len(events)}')
print(f'Post annotation files: {len(post_jsons)}')
print(f'Pre images           : {len(pre_imgs)}')
print(f'Post images          : {len(post_imgs)}')
print()
for ev in sorted(events):
    n = sum(1 for p in post_jsons if ev in p.stem)
    print(f'  {ev}: {n} tiles')

# Inspect one annotation
if post_jsons:
    sample   = _json.loads(post_jsons[0].read_text())
    features = sample.get('features', {}).get('lng_lat', [])
    print()
    print('Sample annotation metadata:', sample.get('metadata', {}))
    print(f'Buildings in sample tile  : {len(features)}')
    if features:
        f0 = features[0]
        print('  uid     :', f0.get('properties', {}).get('uid', '?'))
        print('  subtype :', f0.get('properties', {}).get('subtype', '?'))


In [ ]:
# Visualize one pre/post tile pair
import cv2
import matplotlib.pyplot as plt
from PIL import Image

if post_jsons:
    sample_post_json = post_jsons[0]
    stem = sample_post_json.stem.replace('_post_disaster', '')
    img_dir  = sample_post_json.parent.parent / 'images'
    pre_img  = img_dir / f'{stem}_pre_disaster.png'
    post_img = img_dir / f'{stem}_post_disaster.png'

    if pre_img.exists() and post_img.exists():
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        axes[0].imshow(Image.open(pre_img)); axes[0].set_title(f'Pre-Disaster: {stem}'); axes[0].axis('off')
        axes[1].imshow(Image.open(post_img)); axes[1].set_title(f'Post-Disaster: {stem}'); axes[1].axis('off')
        plt.tight_layout(); plt.show()
    else:
        print('Sample images not found at:', img_dir)

## Section 7 – Building Extraction and Paired Crop Generation

In [ ]:
import re
from tqdm import tqdm
from collections import Counter

def extract_event_from_stem(stem):
    m = re.match(r'^(.+?)_(\d{8})$', stem)
    return m.group(1) if m else stem.split('_')[0]

def collect_all_pairs(xbd_root, label_map=None):
    if label_map is None: label_map = XBD_LABEL_MAP
    xbd_root = Path(xbd_root)

    # Detect layout
    flat_images = xbd_root / 'images'
    flat_labels = xbd_root / 'labels'
    layout = 'flat' if (flat_images.is_dir() and flat_labels.is_dir()) else 'event_subdirs'
    print(f'Layout: {layout}')

    post_jsons = sorted(xbd_root.rglob('*_post_disaster.json'))
    all_records = []

    for post_json in tqdm(post_jsons, desc='Parsing annotations'):
        stem     = post_json.stem.replace('_post_disaster', '')
        pre_json = post_json.parent / f'{stem}_pre_disaster.json'
        if not pre_json.exists(): continue

        # Locate images based on layout
        if layout == 'flat':
            img_dir  = xbd_root / 'images'
            disaster = extract_event_from_stem(stem)
        else:
            img_dir  = post_json.parent.parent / 'images'
            disaster = post_json.parts[-3] if len(post_json.parts) >= 3 else 'unknown'

        pre_img  = img_dir / f'{stem}_pre_disaster.png'
        post_img = img_dir / f'{stem}_post_disaster.png'
        for ext in ['.tif', '.jpg']:
            if not pre_img.exists():  pre_img  = img_dir / f'{stem}_pre_disaster{ext}'
            if not post_img.exists(): post_img = img_dir / f'{stem}_post_disaster{ext}'

        if not (pre_img.exists() and post_img.exists()): continue

        matched = parse_image_pair(pre_json, post_json)
        for b in matched:
            b.update({'pre_image_path': str(pre_img), 'post_image_path': str(post_img),
                      'tile_id': stem, 'disaster_event': disaster})
        all_records.extend(matched)

    return all_records

print('Collecting xBD building pairs ...')
all_records = collect_all_pairs(XBD_ROOT)

if not all_records:
    raise RuntimeError('No building samples found! Check XBD_ROOT path and dataset structure.')

print(f'Total valid building samples: {len(all_records)}')
counts = Counter(r['damage_class'] for r in all_records)
print('\nClass Distribution:')
for i, name in enumerate(CLASS_NAMES):
    cnt = counts.get(i, 0)
    pct = cnt / len(all_records) * 100
    print(f'  {name:<18} : {cnt:>6} ({pct:.1f}%)')


In [ ]:
# Extract paired building crops from tiles
import cv2
from PIL import Image as PILImage

def load_rgb(path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None else None

def extract_crop(image, bbox, pad=4, size=128):
    H, W = image.shape[:2]
    x1,y1,x2,y2 = bbox
    x1,y1 = max(0,x1-pad), max(0,y1-pad)
    x2,y2 = min(W,x2+pad), min(H,y2+pad)
    if x2<=x1 or y2<=y1: return np.zeros((size,size,3),dtype=np.uint8)
    crop = image[y1:y2, x1:x2]
    interp = cv2.INTER_AREA if crop.shape[0]>size else cv2.INTER_LINEAR
    return cv2.resize(crop, (size,size), interpolation=interp)

cache = {}
valid_records = []

for rec in tqdm(all_records, desc='Extracting crops'):
    pp  = rec['pre_image_path']
    qp  = rec['post_image_path']
    if pp not in cache:  cache[pp]  = load_rgb(pp)
    if qp not in cache:  cache[qp]  = load_rgb(qp)
    pre_img  = cache[pp]
    post_img = cache[qp]
    if pre_img is None or post_img is None: continue
    pre_crop  = extract_crop(pre_img,  rec['pre_bbox'],  size=IMAGE_SIZE)
    post_crop = extract_crop(post_img, rec['post_bbox'], size=IMAGE_SIZE)
    if pre_crop.size==0 or post_crop.size==0: continue
    new_rec = dict(rec)
    new_rec['pre_crop']  = pre_crop
    new_rec['post_crop'] = post_crop
    valid_records.append(new_rec)

print(f'Valid paired crops extracted: {len(valid_records)} / {len(all_records)}')

In [ ]:
# Visualize sample building crops
import random
random.seed(42)
sample_idxs = random.sample(range(len(valid_records)), min(8, len(valid_records)))

fig, axes = plt.subplots(2, 8, figsize=(20, 5))
for col, idx in enumerate(sample_idxs):
    rec  = valid_records[idx]
    cls  = CLASS_NAMES[rec['damage_class']]
    col_hex = list(DAMAGE_HEX.values())[rec['damage_class']]
    axes[0][col].imshow(rec['pre_crop'])
    axes[0][col].set_title('Pre', fontsize=7); axes[0][col].axis('off')
    axes[1][col].imshow(rec['post_crop'])
    axes[1][col].set_title(cls[:8], fontsize=7, color=col_hex)
    axes[1][col].axis('off')
plt.suptitle('Sample Building Crop Pairs (Pre / Post)', fontsize=11)
plt.tight_layout(); plt.show()

## Section 8 – Train/Val/Test Split and DataLoaders

In [ ]:
import numpy as np

rng = np.random.default_rng(RANDOM_SEED)

# Event-level split to prevent data leakage
events = {}
for r in valid_records:
    ev = r.get('disaster_event', 'unknown')
    events.setdefault(ev, []).append(r)

event_names = sorted(events.keys())
rng.shuffle(event_names)
n = len(event_names)
n_train = max(1, int(n * TRAIN_RATIO))
n_val   = max(1, int(n * VAL_RATIO))

if n >= 3:
    train_events = event_names[:n_train]
    val_events   = event_names[n_train:n_train+n_val]
    test_events  = event_names[n_train+n_val:]
    train_recs = [r for ev in train_events for r in events[ev]]
    val_recs   = [r for ev in val_events   for r in events[ev]]
    test_recs  = [r for ev in test_events  for r in events[ev]]
    print('Event-level split used (prevents data leakage)')
else:
    idx = np.arange(len(valid_records)); rng.shuffle(idx)
    n_tr = int(len(valid_records)*TRAIN_RATIO); n_vl = int(len(valid_records)*VAL_RATIO)
    train_recs = [valid_records[i] for i in idx[:n_tr]]
    val_recs   = [valid_records[i] for i in idx[n_tr:n_tr+n_vl]]
    test_recs  = [valid_records[i] for i in idx[n_tr+n_vl:]]
    print('Record-level split used (too few events for event-level split)')

print(f'Train: {len(train_recs)}  Val: {len(val_recs)}  Test: {len(test_recs)}')

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from collections import Counter

train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

val_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class XBDDataset(Dataset):
    def __init__(self, records, transform):
        self.records   = records
        self.transform = transform
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        r = self.records[idx]
        return self.transform(r['pre_crop']), self.transform(r['post_crop']), int(r['damage_class']), r.get('uid',str(idx))

# Class-weighted sampler
counts = Counter(r['damage_class'] for r in train_recs)
total  = sum(counts.values())
cw     = {c: total/(NUM_CLASSES*max(counts.get(c,1),1)) for c in range(NUM_CLASSES)}
sample_w = [cw[r['damage_class']] for r in train_recs]
sampler  = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

train_loader = DataLoader(XBDDataset(train_recs, train_tf), batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(XBDDataset(val_recs,   val_tf),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(XBDDataset(test_recs,  val_tf),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'DataLoaders ready – train_batches={len(train_loader)}  val_batches={len(val_loader)}  test_batches={len(test_loader)}')

## Section 9 – Initialize Siamese CNN

In [ ]:
model = SiameseDamageClassifier(num_classes=NUM_CLASSES, dropout=DROPOUT_RATE,
                                 pretrained=True, freeze_backbone=True).to(device)

total_params    = sum(p.numel() for p in model.parameters())
trainable       = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters   : {total_params:,}')
print(f'Trainable (phase 1): {trainable:,}  ({trainable/total_params*100:.1f}%)')
print(f'Frozen (backbone)  : {total_params-trainable:,}')

# Class-weighted loss
cw_tensor   = torch.tensor([cw[i] for i in range(NUM_CLASSES)], dtype=torch.float32).to(device)
criterion   = torch.nn.CrossEntropyLoss(weight=cw_tensor)
optimizer   = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                 lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
use_amp     = device.type == 'cuda'
if use_amp:
    from torch.cuda.amp import GradScaler
    scaler = GradScaler()
else:
    scaler = None
print('Model ready. Training on:', device)

## Section 10 – Training Loop

In [ ]:
import time
from torch.cuda.amp import autocast

history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_val_loss = float('inf')
patience_cnt  = 0
best_ckpt     = CHECKPOINT_DIR / 'best_model.pth'
start_time    = time.time()

for epoch in range(1, NUM_EPOCHS+1):
    # Phase 2: unfreeze backbone after epoch 3
    if epoch == 4:
        print('  → Unfreezing backbone for fine-tuning (Phase 2) ...')
        model.unfreeze_backbone()
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE/5, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS-epoch+1)

    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    t0 = time.time()
    for pre, post, labels, _ in tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [Train]', leave=False):
        pre, post, labels = pre.to(device), post.to(device), labels.to(device)
        optimizer.zero_grad()
        if use_amp:
            with autocast():
                logits = model(pre, post); loss = criterion(logits, labels)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        else:
            logits = model(pre, post); loss = criterion(logits, labels)
            loss.backward(); optimizer.step()
        tr_loss    += loss.item() * labels.size(0)
        tr_correct += (logits.argmax(1)==labels).sum().item()
        tr_total   += labels.size(0)

    # ── Validate ─────────────────────────────────────────────────────────────
    model.eval()
    vl_loss, vl_correct, vl_total = 0.0, 0, 0
    with torch.no_grad():
        for pre, post, labels, _ in val_loader:
            pre, post, labels = pre.to(device), post.to(device), labels.to(device)
            logits = model(pre, post); loss = criterion(logits, labels)
            vl_loss    += loss.item() * labels.size(0)
            vl_correct += (logits.argmax(1)==labels).sum().item()
            vl_total   += labels.size(0)

    scheduler.step()
    tr_l = tr_loss/tr_total; tr_a = tr_correct/tr_total
    vl_l = vl_loss/vl_total; vl_a = vl_correct/vl_total
    history['train_loss'].append(tr_l); history['val_loss'].append(vl_l)
    history['train_acc'].append(tr_a);  history['val_acc'].append(vl_a)
    elapsed = time.time()-t0

    print(f'Epoch {epoch:>2}/{NUM_EPOCHS}  '
          f'train_loss={tr_l:.4f}  train_acc={tr_a:.4f}  '
          f'val_loss={vl_l:.4f}  val_acc={vl_a:.4f}  ({elapsed:.1f}s)')

    # ── Checkpoint ───────────────────────────────────────────────────────────
    if vl_l < best_val_loss:
        best_val_loss = vl_l; patience_cnt = 0
        torch.save({'epoch':epoch,'model_state_dict':model.state_dict(),
                    'val_loss':vl_l,'val_acc':vl_a}, best_ckpt)
        # Backup to Drive
        try:
            import shutil
            shutil.copy2(best_ckpt, DRIVE_SAVE_DIR/'best_model.pth')
        except: pass
        print(f'  ✓ Best checkpoint saved  val_loss={vl_l:.4f}')
    else:
        patience_cnt += 1
        if patience_cnt >= EARLY_STOP_PATIENCE:
            print(f'Early stopping at epoch {epoch} (patience={EARLY_STOP_PATIENCE})')
            break

total_time = time.time()-start_time
print(f'\nTraining complete: {total_time/60:.1f} minutes')

## Section 11 – Training Curves

In [ ]:
import matplotlib.pyplot as plt
epochs = range(1, len(history['train_loss'])+1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train Loss')
axes[0].plot(epochs, history['val_loss'],   'r-o', label='Val Loss')
axes[0].set_title('Loss Curve'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('CE Loss')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs, history['train_acc'], 'b-o', label='Train Acc')
axes[1].plot(epochs, history['val_acc'],   'r-o', label='Val Acc')
axes[1].set_title('Accuracy Curve'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True)

plt.suptitle('DisasterVision – Training History', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

## Section 12 – Evaluation on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# Load best checkpoint
ckpt = torch.load(best_ckpt, map_location=device, weights_only=True)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded best checkpoint (epoch {ckpt["epoch"]}, val_loss={ckpt["val_loss"]:.4f})')

all_preds, all_labels = [], []
with torch.no_grad():
    for pre, post, labels, _ in tqdm(test_loader, desc='Evaluating'):
        pre, post = pre.to(device), post.to(device)
        logits    = model(pre, post)
        preds     = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())

print(f'\nTest samples: {len(all_labels)}')
print('\n' + classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

# Save report
report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4)
(OUTPUT_DIR/'classification_report.txt').write_text(report)
print('Saved classification_report.txt')

## Section 13 – Confusion Matrix

In [ ]:
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, colorbar=True, cmap='Blues', values_format='d')
ax.set_title('DisasterVision – 4-Class Damage Confusion Matrix', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved confusion_matrix.png')

## Section 14 – Class Distribution

In [ ]:
counts_all = Counter(r['damage_class'] for r in valid_records)
vals   = [counts_all.get(i,0) for i in range(NUM_CLASSES)]
colors = ['#2ecc71','#f1c40f','#e67e22','#e74c3c']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(CLASS_NAMES, vals, color=colors, edgecolor='black', alpha=0.9)
ax.bar_label(bars, fmt='%d', padding=3, fontsize=11)
ax.set_title('Full Dataset – Class Distribution', fontsize=13)
ax.set_ylabel('Sample Count'); ax.set_xlabel('Damage Category')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved class_distribution.png')

## Section 15 – Damage Visualization

In [ ]:
# Visualize predictions on a sample tile
sample_post_json = post_jsons[0]
stem    = sample_post_json.stem.replace('_post_disaster', '')
pre_img_path  = sample_post_json.parent.parent / 'images' / f'{stem}_pre_disaster.png'
post_img_path = sample_post_json.parent.parent / 'images' / f'{stem}_post_disaster.png'

if pre_img_path.exists() and post_img_path.exists():
    pre_img  = load_rgb(pre_img_path)
    post_img = load_rgb(post_img_path)
    H, W = pre_img.shape[:2]

    pre_json_path  = sample_post_json.parent / f'{stem}_pre_disaster.json'
    matched_tile   = parse_image_pair(pre_json_path, sample_post_json, W, H)

    # Predict each building
    damage_color_bgr = {0:(46,204,113), 1:(241,196,15), 2:(230,126,34), 3:(231,76,60)}
    model.eval()
    pred_vis = cv2.cvtColor(post_img.copy(), cv2.COLOR_RGB2BGR)
    overlay  = pred_vis.copy()

    for b in matched_tile:
        pre_c  = extract_crop(pre_img,  b['pre_bbox'],  size=IMAGE_SIZE)
        post_c = extract_crop(post_img, b['post_bbox'], size=IMAGE_SIZE)
        with torch.no_grad():
            pt = val_tf(pre_c).unsqueeze(0).to(device)
            qt = val_tf(post_c).unsqueeze(0).to(device)
            logits = model(pt, qt)
            pred_cls = int(logits.argmax(1).item())
        color = damage_color_bgr[pred_cls]
        poly  = b['post_polygon'].reshape(-1,1,2)
        cv2.fillPoly(overlay, [poly], color)
        cv2.polylines(pred_vis, [poly], True, color, 2)

    cv2.addWeighted(overlay, 0.35, pred_vis, 0.65, 0, pred_vis)
    pred_vis_rgb = cv2.cvtColor(pred_vis, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(post_img); axes[0].set_title('Post-Disaster (Original)'); axes[0].axis('off')
    axes[1].imshow(pred_vis_rgb); axes[1].set_title('Predicted Damage Overlay'); axes[1].axis('off')
    # Legend
    import matplotlib.patches as mpatches
    patches = [mpatches.Patch(color=DAMAGE_HEX[i], label=CLASS_NAMES[i]) for i in range(4)]
    axes[1].legend(handles=patches, loc='upper right', fontsize=9)
    plt.suptitle(f'Building Damage Visualization: {stem}', fontsize=13)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'damage_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved damage_visualization.png')
else:
    print('Sample tile images not found – skipping visualization')

## Section 16 – Save Evaluation Metrics

In [ ]:
import json as _json
from sklearn.metrics import f1_score, precision_score, recall_score

metrics = {
    'accuracy':         float(accuracy_score(all_labels, all_preds)),
    'macro_f1':         float(f1_score(all_labels, all_preds, average='macro', zero_division=0)),
    'weighted_f1':      float(f1_score(all_labels, all_preds, average='weighted', zero_division=0)),
    'macro_precision':  float(precision_score(all_labels, all_preds, average='macro', zero_division=0)),
    'macro_recall':     float(recall_score(all_labels, all_preds, average='macro', zero_division=0)),
    'num_test_samples': len(all_labels),
    'class_names':      CLASS_NAMES,
    'per_class_f1':     f1_score(all_labels, all_preds, average=None, zero_division=0).tolist(),
    'training_minutes': total_time/60,
    'total_samples':    len(valid_records),
    'train_samples':    len(train_recs),
    'val_samples':      len(val_recs),
    'test_samples':     len(test_recs),
}

(OUTPUT_DIR/'evaluation_metrics.json').write_text(_json.dumps(metrics, indent=2))
(OUTPUT_DIR/'training_history.json').write_text(_json.dumps(history, indent=2))

print('Evaluation Metrics:')
for k,v in metrics.items():
    if isinstance(v, float): print(f'  {k:<22}: {v:.4f}')
    elif isinstance(v, int):  print(f'  {k:<22}: {v}')

## Section 17 – Model Export and Download

In [ ]:
import shutil, zipfile

# Save model config
model_config = {
    'architecture': 'SiameseDamageClassifier',
    'backbone': 'resnet18',
    'num_classes': NUM_CLASSES,
    'feature_dim': 512,
    'fusion': 'concat(pre, post, abs_diff)',
    'image_size': IMAGE_SIZE,
}
class_mapping = {'classes': CLASS_NAMES, 'label_map': XBD_LABEL_MAP}
preproc_cfg   = {'image_size': IMAGE_SIZE, 'mean':[0.485,0.456,0.406], 'std':[0.229,0.224,0.225]}

(CHECKPOINT_DIR/'model_config.json').write_text(_json.dumps(model_config, indent=2))
(CHECKPOINT_DIR/'class_mapping.json').write_text(_json.dumps(class_mapping, indent=2))
(CHECKPOINT_DIR/'preprocessing_config.json').write_text(_json.dumps(preproc_cfg, indent=2))

# Create export ZIP
zip_path = '/content/DisasterVision_Model_Export.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in CHECKPOINT_DIR.iterdir():
        zf.write(f, f'checkpoints/{f.name}')
    for f in OUTPUT_DIR.iterdir():
        zf.write(f, f'outputs/{f.name}')

print(f'Export ZIP created: {zip_path}')

# Copy to Drive
try:
    shutil.copy2(zip_path, DRIVE_SAVE_DIR/'DisasterVision_Model_Export.zip')
    print(f'ZIP also saved to: {DRIVE_SAVE_DIR}/DisasterVision_Model_Export.zip')
except Exception as e:
    print(f'Could not save to Drive: {e}')

In [ ]:
# Download the model export ZIP to your local machine
from google.colab import files
files.download('/content/DisasterVision_Model_Export.zip')
print('Download initiated. Extract to DisasterVision/checkpoints/ on your Windows machine.')
print()
print('Windows deployment:')
print('  1. Expand-Archive DisasterVision_Model_Export.zip -DestinationPath .')
print('  2. streamlit run app.py')